# 06 — Large images, sliding windows, and a Sentinel-2 tile

DOTA models are trained on 1024×1024 tiles, but real scenes are thousands of pixels wide and come from sensors the model never saw. This lesson covers both problems: sliding-window inference on a large image, then zero-shot ship detection on a 10 m Copernicus Sentinel-2 tile.

| | |
|---|---|
| Needs | Internet (two ≈330 MB checkpoints) |
| Kaggle | Yes — Internet ON, GPU recommended (the Sentinel-2 run is 49 windows) |
| Posts | [The final OrientedDet pretrained model](https://deeplearning.earth/posts/2026-06-29_announcing_the_final_oriented_det_pretrained_model/) · [Zero-shot ship detection on a Copernicus Sentinel-2 tile](https://deeplearning.earth/posts/2026-06-25_zero-shot_ship_detection_on_a_copernicus_sentinel-2_tile_with_oriented_rcnn/) |

## Why windows?

Shrinking a 4,000 px scene to the 1024 px canvas would turn a 20 px car into a 5 px smudge. Instead, inference cuts the image into canvas-sized **windows at native resolution**, runs each one, shifts the boxes back into scene coordinates, and merges everything with rotated NMS.

Windows **overlap** (200 px by default, the same overlap the training tiles use), so an object cut in half by one window's edge appears whole in its neighbour. The duplicate copies are removed by the final NMS. `ignore_margin_pixels` (off by default) additionally drops detections whose centre falls in the overlap band of an interior edge. The last row and column of windows are flushed to the image border, so every window contains real pixels.

Checkpoints trained with `keep_ratio` or `pad` resizing (HRSC2016, SSDD, HRSID) do not tile at all: they rescale the whole image, as in training (lesson 08).

## Setup

The same two cells open every lesson. The first installs OrientedDet: your editable checkout if this notebook sits inside the repo, otherwise the PyPI wheel. The second sets paths and the device, and defines the small helpers used across the series (`hub_files`, `hub_model`, `detect`, `show`, ...). On Kaggle, switch **Internet ON** in the notebook settings.

In [ ]:
# Install: editable checkout when this notebook lives inside the repo, else PyPI.
import importlib.util
from pathlib import Path

REPO = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "pyproject.toml").is_file() and (p / "oriented_det").is_dir()), None)
_spec = importlib.util.find_spec("oriented_det")
if REPO is None:
    %pip install -q "oriented-det>=0.3.1"
elif _spec is None or not Path(_spec.origin).resolve().is_relative_to(REPO.resolve()):
    %pip install -q -e "{REPO}"

In [ ]:
import os, subprocess, sys, time, urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import oriented_det

GH_RAW = "https://raw.githubusercontent.com/DL4EO/oriented-det/main"
IN_KAGGLE = Path("/kaggle/working").is_dir()
HAS_REPO = REPO is not None
WORK = (Path("/kaggle/working") if IN_KAGGLE else
        Path(os.environ.get("ORIENTED_DET_TUTORIAL_WORK", (REPO or Path.cwd()) / "work_dirs")).resolve())
WORK.mkdir(parents=True, exist_ok=True)
os.environ["ORIENTED_DET_PROJECT_ROOT"] = str(WORK)  # runs/ and predictions/ land here
if not HAS_REPO:  # wheel install: keep Hub weights out of site-packages
    os.environ.setdefault("ORIENTED_DET_PRETRAINED_DIR", str(WORK / "pretrained"))

CONFIGS = Path(oriented_det.__file__).parent / "configs"  # recipes shipped with the package
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
ODET = [sys.executable, "-m", "oriented_det.cli"]  # same as typing `odet` in a shell


def run(cmd):
    """Run a CLI command, echoing it first."""
    cmd = [str(c) for c in cmd]
    print("$", " ".join(["odet", *cmd[3:]] if cmd[:3] == ODET else cmd), flush=True)
    subprocess.run(cmd, check=True)


def demo_file(rel):
    """demo/<rel> from the checkout, else downloaded once from GitHub into WORK/demo."""
    if HAS_REPO and (REPO / "demo" / rel).is_file():
        return REPO / "demo" / rel
    target = WORK / "demo" / rel
    if not target.is_file():
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(f"{GH_RAW}/demo/{rel}", target)
    return target


def kaggle_input(*needles):
    """First /kaggle/input/<dataset> whose name contains one of `needles` (None off Kaggle)."""
    root = Path("/kaggle/input")
    dirs = sorted(p for p in root.iterdir() if p.is_dir()) if root.is_dir() else []
    return next((p for n in needles for p in dirs if n.lower() in p.name.lower()), None)


def hub_files(slug):
    """Download a Hub checkpoint and its sidecar config once -> (checkpoint_path, config_path)."""
    from huggingface_hub import hf_hub_download
    from oriented_det.pretrained import ensure_checkpoint, load_manifest, resolve_checkpoint_sidecar_config

    ckpt = ensure_checkpoint(f"hf://{slug}")
    manifest = load_manifest()
    sidecar = resolve_checkpoint_sidecar_config(ckpt) or hf_hub_download(
        manifest["repo_id"], ckpt.with_suffix(".json").name,
        revision=manifest.get("revision", "main"), local_dir=ckpt.parent)
    return ckpt, Path(sidecar)


def hub_model(slug, device=DEVICE):
    """Hub checkpoint -> (model, config, class_names). Lesson 01 unpacks it."""
    from oriented_det.runtime.checkpoint import load_model_from_checkpoint

    ckpt, sidecar = hub_files(slug)
    return load_model_from_checkpoint(str(ckpt), str(sidecar), device=device)


def detect(model, config, class_names, image, score_thr=None, classes=None):
    """Detections on a PIL image with the checkpoint's deploy settings (sliding windows past the canvas)."""
    from oriented_det.runtime.inference import apply_nms_to_detections, run_inference_auto
    from oriented_det.train.config import (get_preprocessing_params, resolve_inference_score_threshold,
                                           resolve_inference_sliding_window_overlap_pixels)

    nms = config.production.final_nms_iou_threshold
    dets = run_inference_auto(
        image=image, model=model, device=DEVICE, preprocessing=get_preprocessing_params(config),
        score_threshold=resolve_inference_score_threshold(config) if score_thr is None else score_thr,
        nms_threshold=nms, overlap_pixels=resolve_inference_sliding_window_overlap_pixels(config),
        class_names=class_names)
    dets = apply_nms_to_detections(dets, iou_threshold=nms)
    if classes:
        dets = [d for d in dets if class_names[int(d["label"]) - 1] in classes]
    return dets


def show(*images, titles=(), size=6):
    """Display images side by side."""
    _, axes = plt.subplots(1, len(images), figsize=(size * len(images), size), squeeze=False)
    for ax, img, title in zip(axes[0], images, [*titles, *[""] * len(images)]):
        ax.imshow(img)
        ax.set_title(title)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


print(f"oriented_det: {Path(oriented_det.__file__).parent}")
print(f"device={DEVICE}  kaggle={IN_KAGGLE}  work={WORK}")

## 1. The window grid

Along each axis, window origins step by `canvas − overlap`, and the last origin is moved back so the final window ends exactly at the border. Five lines reproduce the rule; the library's `count_sliding_window_positions` confirms the count.

In [ ]:
from PIL import Image
from matplotlib.patches import Rectangle
from oriented_det.runtime.inference import count_sliding_window_positions


def window_origins(length, canvas=1024, overlap=200):
    if length <= canvas:
        return [0]
    origins = list(range(0, length, canvas - overlap))
    origins[-1] = length - canvas
    return sorted(set(origins))


large = Image.open(demo_file("large.jpg")).convert("RGB")
W, H = large.size
pre = {"resize_mode": "fixed", "target_size": (1024, 1024)}
for overlap in (0, 200, 512):
    n = len(window_origins(W, overlap=overlap)) * len(window_origins(H, overlap=overlap))
    print(f"overlap {overlap:3d} px: {n} windows (library says {count_sliding_window_positions(H, W, pre, overlap_pixels=overlap)})")

fig, ax = plt.subplots(figsize=(5, 7))
ax.imshow(large)
for i, (x0, y0) in enumerate([(x, y) for y in window_origins(H) for x in window_origins(W)]):
    ax.add_patch(Rectangle((x0, y0), 1024, 1024, fill=False, lw=2, color=plt.cm.tab10(i)))
ax.set_title(f"{W}x{H} px, 200 px overlap"); ax.axis("off"); plt.show()

## 2. Detect on the full image

`detect()` (from the setup cell) calls `run_inference_auto`, which notices that the image is larger than the canvas and switches to sliding windows with the checkpoint's `production.overlap_pixels`.

In [ ]:
from collections import Counter
from oriented_det.runtime.inference import visualize_results

model, config, class_names = hub_model("oriented_rcnn_dota_le90_1x")
t0 = time.perf_counter()
dets = detect(model, config, class_names, large)
print(f"{len(dets)} detections in {time.perf_counter() - t0:.1f} s")
print(Counter(class_names[d["label"] - 1] for d in dets).most_common())
vis = visualize_results(large.copy(), dets, class_names)
show(vis, vis.crop((0, 700, 800, 1300)), titles=("full image", "zoom"), size=7)

## 3. What does overlap buy?

Run the same image with 0 px and 200 px overlap and look for boxes that only the overlapping run finds.

In [ ]:
from oriented_det.ops import rbox_iou
from oriented_det.runtime.inference import apply_nms_to_detections, run_inference_auto
from oriented_det.train.config import get_preprocessing_params


def detect_with_overlap(px):
    out = run_inference_auto(image=large, model=model, device=DEVICE, preprocessing=get_preprocessing_params(config),
                             score_threshold=config.production.score_threshold, nms_threshold=0.1,
                             overlap_pixels=px, class_names=class_names)
    return apply_nms_to_detections(out, iou_threshold=0.1)


no_overlap, with_overlap = detect_with_overlap(0), detect_with_overlap(200)
only_with = [d for d in with_overlap if max((rbox_iou(d["rbox"], e["rbox"]) for e in no_overlap), default=0) < 0.3]
print(f"0 px overlap: {len(no_overlap)} detections, 200 px: {len(with_overlap)}, found only with overlap: {len(only_with)}")

On this image the difference is small: it needs only 2×3 windows, and flushing the last window to the border already creates wide overlaps. On a 10,000 px scene with dozens of interior seams, the objects cut by a seam are exactly the ones overlap rescues, and the extra windows are the price.

## 4. Domain transfer: ships on Sentinel-2

Sentinel-2's true-colour product has a 10 m ground sample distance. DOTA imagery is sub-metre to a few metres per pixel, so a 100 m ship that spans 100+ px in DOTA is about 10 px here. Three knobs close part of the gap:

- `--zoom 4` upsamples the tile before inference so ships approach the scale the model learned (a 1024 px tile becomes 4096 px, i.e. 49 windows at 512 px overlap);
- `--classes ship` keeps only the class we care about;
- `--score-thr 0.15` accepts lower confidence, because a model outside its domain is less sure of itself.

These are the settings from the post, which used the 3× Oriented R-CNN. The CLI writes a PNG and, with `--json-per-image`, a JSON file of boxes in original-tile coordinates.

In [ ]:
import json

S2_SLUG = "oriented_rcnn_dota_le90_3x"  # the 1x checkpoint also works and skips a download
hub_files(S2_SLUG)  # the CLI finds the sidecar config next to the checkpoint
tile = demo_file("copernicus/T30NZM_20260616T101021_TCI_10m_2976_7936.png")
out = WORK / "06_s2_ships.png"
run([*ODET, "image-demo", tile, f"hf://{S2_SLUG}", "--out-file", out, "--device", DEVICE,
     "--score-thr", 0.15, "--nms-thr", 0.2, "--classes", "ship", "--zoom", 4,
     "--overlap-pixels", 512, "--ignore-margin-pixels", 256, "--json-per-image"])

result = json.loads(out.with_suffix(".json").read_text())
ships = result["detections"]
lengths = [max(d["rbox"]["width"], d["rbox"]["height"]) * 10 for d in ships]  # 10 m per pixel
print(f"{len(ships)} ships; inference size {result['inference_size']}")
if lengths:
    print(f"estimated lengths: median {sorted(lengths)[len(lengths) // 2]:.0f} m, max {max(lengths):.0f} m")
show(Image.open(tile), Image.open(out), titles=("Sentinel-2 TCI, 10 m", f"{len(ships)} ships (zoom x4)"), size=7)

The box length times the ground sample distance gives a rough ship length: the kind of measurement an oriented box makes possible and an axis-aligned box does not. Expect some false positives on coastlines, wakes, and clouds: the model never saw 10 m imagery during training. Fine-tuning on a SAR or medium-resolution dataset (lessons 08–10) is the real fix.

## What to try next

- Try `--zoom 2` and `--zoom 1` on the Sentinel-2 tile and count ships: where does the model stop seeing them?
- Add `harbor` to `--classes`, or drop the filter entirely to see what else the model hallucinates at 10 m.
- Training-time tiling uses the same idea: `odet tile-dota` (lesson 09) and `oriented_det.data.tiling.ImageTiler` cut labelled scenes into overlapping tiles and clip the boxes.
- **Next:** [07 — Recipes, training knobs, and honest evaluation](07_config_train_eval.ipynb).